# 03.08 - DeepLabV3 ResNet50 transfer learning

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** DeepLabV3-ResNet50 head-training and validation evidence.

Adapt the official DeepLabV3 ResNet50 architecture to three target classes, train its segmentation heads on a compact split, and evaluate pixel accuracy and per-class IoU.

## Core Ideas

For multiclass semantic segmentation, logits are `[N,C,H,W]` and CrossEntropyLoss targets are `int64 [N,H,W]`. A pretrained model must have both the main classifier and auxiliary classifier replaced. Head-only training freezes the backbone and can use a larger head learning rate. Validation masks remain untouched.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

SEED = 3
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Prepared Three-Class Segmentation Split

Nine 32×32 RGB images contain background class 0 and one foreground class (1 or 2). Six images train and three disjoint images validate.

In [ ]:
all_images = torch.full((9, 3, 32, 32), 0.1, dtype=torch.float32)
all_masks = torch.zeros((9, 32, 32), dtype=torch.int64)
for index in range(9):
    class_id = 1 + index % 2
    x1, y1 = 4 + index % 5, 6 + (index * 2) % 6
    all_masks[index, y1:y1 + 14, x1:x1 + 12] = class_id
    all_images[index, class_id, y1:y1 + 14, x1:x1 + 12] = 0.95
train_images, validation_images = all_images[:6], all_images[6:]
train_masks, validation_masks = all_masks[:6], all_masks[6:]
print("split:", train_images.shape, validation_images.shape, "validation labels:", torch.unique(validation_masks).tolist())

## Exercise 03-A: Configure target-class heads

Support optional official weights, replace both classifiers when pretrained, and freeze the backbone for head-only practice.

**Return structure — `build_transfer_deeplab50`:** A Torchvision `DeepLabV3` on `device` with main and auxiliary outputs containing `num_classes` channels. Backbone parameters are frozen; classifier parameters are trainable.

In [ ]:
# TODO 03-A
def build_transfer_deeplab50(num_classes=3, use_pretrained=False, device=DEVICE):
    raise NotImplementedError("Complete Exercise 03-A")


# Smoke check: configure the exact architecture offline.
transfer_model = build_transfer_deeplab50()
print(transfer_model.classifier[-1], transfer_model.aux_classifier[-1])

## Exercise 03-B: Create head parameter groups

Select only trainable main and auxiliary classifier parameters for the optimizer.

**Return structure — `segmentation_head_groups`:** A `list[dict]` of two groups; each has `params` (non-empty list) and float `lr`. No frozen backbone parameter appears.

In [ ]:
# TODO 03-B
def segmentation_head_groups(model, main_lr=0.01, auxiliary_lr=0.005):
    raise NotImplementedError("Complete Exercise 03-B")


# Smoke check: inspect optimizer groups.
head_groups = segmentation_head_groups(transfer_model)
print([group["lr"] for group in head_groups], [sum(p.numel() for p in group["params"]) for group in head_groups])

## Exercise 03-C: Train the segmentation heads

Keep the frozen backbone in evaluation mode, optimize main plus weighted auxiliary loss, and use the complete training split.

**Return structure — `train_deeplab_heads`:** A `list[dict]` of length `epochs`; each row has integer `epoch` and Python floats `main_loss`, `aux_loss`, `total_loss`, and `runtime_seconds`. The model is updated.

In [ ]:
# TODO 03-C
def train_deeplab_heads(model, images, masks, epochs=2, device=DEVICE):
    raise NotImplementedError("Complete Exercise 03-C")


# Smoke check and complete-training-split evidence.
training_history = train_deeplab_heads(transfer_model, train_images, train_masks)
print(pd.DataFrame(training_history).to_string(index=False))

## Exercise 03-D: Evaluate pixel metrics

Use only the main `out` logits and calculate IoU for every documented class on all validation pixels.

**Return structure — `evaluate_deeplab`:** A dictionary with `predictions` as CPU int64 `[N,H,W]`, Python float `pixel_accuracy`, `per_class_iou` as `list[float]` length `num_classes`, Python float `mean_iou`, and integer `validation_pixels`.

In [ ]:
# TODO 03-D
def evaluate_deeplab(model, images, masks, num_classes=3, device=DEVICE):
    raise NotImplementedError("Complete Exercise 03-D")


# Smoke check: evaluate the untouched validation split.
validation_result = evaluate_deeplab(transfer_model, validation_images, validation_masks)
print({key: value for key, value in validation_result.items() if key != "predictions"})

## Test Cases

**Return structure — `run_day03_tests`:** Returns `None`; assertions and `Day 03 tests passed` communicate success.

In [ ]:
def run_day03_tests():
    assert transfer_model.classifier[-1].out_channels == transfer_model.aux_classifier[-1].out_channels == 3
    assert all(not parameter.requires_grad for parameter in transfer_model.backbone.parameters())
    assert len(head_groups) == 2 and [group["lr"] for group in head_groups] == [0.01, 0.005]
    assert len(training_history) == 2 and all(row["total_loss"] > 0 for row in training_history)
    assert validation_result["predictions"].shape == validation_masks.shape
    assert validation_result["predictions"].dtype == torch.int64
    assert len(validation_result["per_class_iou"]) == 3 and validation_result["validation_pixels"] == validation_masks.numel()
    assert 0.0 <= validation_result["pixel_accuracy"] <= 1.0 and 0.0 <= validation_result["mean_iou"] <= 1.0
    print("Day 03 tests passed")


run_day03_tests()

## Day 03 Checklist

- [ ] Replace both main and auxiliary classifiers.
- [ ] Freeze the backbone explicitly.
- [ ] Use int64 `[N,H,W]` masks with CrossEntropyLoss.
- [ ] Evaluate every validation pixel and class.
- [ ] Run the test cases.